# HeatUp — Production Notebook 3: Anharmonic Free-Energy Analysis

Deep-dive into the anharmonic phonon analysis for validated materials:

1. **VDOS inspection** — visualise the anharmonic VDOS for all simulated temperatures, compare with the harmonic phonon DOS, check for soft-mode signatures.
2. **Free-energy decomposition** — break down G(T) = E₀ + F_vib + F_el + F_mag + F_conf using the HeatUp `GibbsAssembler`.
3. **Thermodynamic integrals** — plot E_vib(T), F_vib(T), Cv(T), S_vib(T) at each MD temperature.
4. **Multi-temperature consistency** — compare VDOS and F(T) across different AIMD temperatures to check for phase transitions.
5. **Hull contribution breakdown** — show how each free-energy term shifts the hull distance.

**Prerequisites:** notebook 1 (produces `anharmonic_phonons/` cache folders).

---

## 0 — Imports

In [ ]:
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display

# ── SSE-AL library ────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from library.md_pipeline import scan_database, load_analysis
from library.anharmonic_phonons import (
    compute_anharmonic_phonons_for_sim,
    get_anharmonic_free_energy,
    _find_completed_sim_dirs,
    _load_cached_vdos,
)

# ── HeatUp ───────────────────────────────────────────────────────────────────
HEATUP_ROOT = os.path.abspath(os.path.join('..', '..'))
if HEATUP_ROOT not in sys.path:
    sys.path.insert(0, HEATUP_ROOT)
from heatup.free_energy import (
    GibbsAssembler, build_default_assembler,
    harmonic_f_vib, anharmonic_f_vib,
    electronic_f_el, magnetic_f_mag,
    configurational_f_conf,
)
import heatup.config as cfg

warnings.filterwarnings('ignore')

# Journal-quality plot style
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 9,
    'axes.labelsize': 9, 'axes.titlesize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'legend.fontsize': 7, 'axes.linewidth': 0.7,
    'axes.spines.top': False, 'axes.spines.right': False,
})
C = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73',
     'red':'#D55E00','purple':'#CC79A7','sky':'#56B4E9'}

print('Imports OK')

## 1 — Configuration

In [ ]:
DATABASE_ROOT     = 'database'
HULL_TEMPERATURES = list(range(0, 1501, 50))   # K

# Materials to analyse in detail (set ANALYSE_ALL = True for all database entries)
ANALYSE_ALL   = True
MATERIAL_LIST = [
    ('AgI',      'P6_3mc'),
    ('Li2ZrCl6', 'R-3m'),
]

# Force recompute even if cache exists
FORCE_RECOMPUTE = False

print('Configuration loaded.')

## 2 — Collect all materials to analyse

In [ ]:
if ANALYSE_ALL:
    records = scan_database(DATABASE_ROOT)
    mat_sym_list = list({(r['material'], r['symmetry']) for r in records})
    mat_sym_list = sorted(mat_sym_list)
    print(f'Found {len(mat_sym_list)} unique material/symmetry combinations in database.')
else:
    mat_sym_list = MATERIAL_LIST
    print(f'Analysing {len(mat_sym_list)} specified materials.')

print('\nMaterials:')
for mat, sym in mat_sym_list:
    print(f'  {mat}/{sym}')

## 3 — Compute / load anharmonic VDOS for all materials

In [ ]:
vdos_cache = {}   # (mat, sym, temp_folder) → vdos dict
fe_cache   = {}   # (mat, sym) → free_energy dict

for mat, sym in mat_sym_list:
    sym_dir  = os.path.join(DATABASE_ROOT, mat, sym)
    aimd_dir = os.path.join(sym_dir, 'aimd')

    if not os.path.isdir(aimd_dir):
        continue

    sim_dirs = _find_completed_sim_dirs(aimd_dir)
    if not sim_dirs:
        continue

    print(f'\n  {mat}/{sym}  ({len(sim_dirs)} sim dir(s))')

    for sim_dir in sim_dirs:
        temp_folder = os.path.basename(sim_dir)

        # Compute or load VDOS
        fe = compute_anharmonic_phonons_for_sim(
            sim_dir,
            temperatures    = HULL_TEMPERATURES,
            force_recompute = FORCE_RECOMPUTE,
        )
        if fe is not None:
            # Load the cached VDOS
            vd = _load_cached_vdos(sim_dir)
            if vd is not None:
                vdos_cache[(mat, sym, temp_folder)] = vd
                print(f'    {temp_folder}: VDOS loaded ({len(vd["omega_mev"])} points)')

    # Aggregate free energy across all temperatures
    fe_agg = get_anharmonic_free_energy(
        sym_dir,
        temperatures        = HULL_TEMPERATURES,
        force_recompute     = FORCE_RECOMPUTE,
        run_aimd_if_missing = False,
    )
    if fe_agg is not None:
        fe_cache[(mat, sym)] = fe_agg
        n_src = fe_agg.get('n_sources', 1)
        warn  = '  ⚠ CONSISTENCY WARNING' if fe_agg.get('consistency_warning') else ''
        print(f'    Aggregated F(T) from {n_src} simulation(s){warn}')

print(f'\nLoaded VDOS for {len(vdos_cache)} (material, symmetry, temperature) entries.')

## 4 — VDOS dashboard

For each material: plot all AIMD temperatures side by side.
Highlights soft-mode window and marks the zero-mode fraction ζ.

In [ ]:
def plot_vdos_dashboard(mat, sym):
    entries = [(k, v) for k, v in vdos_cache.items() if k[0] == mat and k[1] == sym]
    if not entries:
        print(f'No VDOS data for {mat}/{sym}')
        return

    n     = len(entries)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.2), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, ((_, _, temp_label), vd) in zip(axes, entries):
        om = np.array(vd['omega_mev'])
        g  = np.array(vd['vdos'])

        ax.fill_between(om, 0, g, alpha=0.5, color=C['blue'])
        ax.plot(om, g, color=C['blue'], lw=0.9)

        # Soft-mode window
        win = cfg.VIB_ZERO_WINDOW_MEV
        ax.axvspan(0, win, color=C['red'], alpha=0.25, label=f'|ω| < {win:.0f} meV')

        # Compute ζ
        mask  = om <= win
        zeta  = float(np.trapz(g[mask], om[mask])) if mask.any() else 0.0
        color = C['red'] if zeta >= cfg.VIB_ZERO_FRAC_FAIL else \
                C['orange'] if zeta >= cfg.VIB_ZERO_FRAC_WARN else C['green']

        ax.text(0.97, 0.96, f'ζ = {zeta * 100:.1f}%',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=8, color=color, fontweight='bold')

        ax.set_xlabel('ω (meV)')
        ax.set_ylabel('VDOS (normalised)' if ax is axes[0] else '')
        ax.set_title(f'{temp_label}')
        ax.set_xlim(om.min(), om.max())
        ax.set_ylim(bottom=0)
        ax.legend(frameon=False, fontsize=6)

    fig.suptitle(f'{mat}/{sym} — Anharmonic VDOS', fontsize=10, fontweight='bold')
    plt.tight_layout()
    out = f'vdos_{mat}_{sym}.pdf'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')


# Plot for each material
for mat, sym in mat_sym_list:
    if any(k[0] == mat and k[1] == sym for k in vdos_cache):
        plot_vdos_dashboard(mat, sym)

## 5 — Compare harmonic DOS vs anharmonic VDOS

In [ ]:
for mat, sym in mat_sym_list:
    sym_dir  = os.path.join(DATABASE_ROOT, mat, sym)
    dos_path = os.path.join(sym_dir, 'phonons', 'dos.json')

    anh_entries = [(k, v) for k, v in vdos_cache.items() if k[0] == mat and k[1] == sym]
    if not anh_entries or not os.path.exists(dos_path):
        continue

    with open(dos_path) as fh:
        dos = json.load(fh)
    om_harm = np.array(dos['energies_eV']) * 1000   # eV → meV
    g_harm  = np.array(dos['weights'])
    norm = np.trapz(g_harm, om_harm)
    if norm > 0:
        g_harm /= norm

    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.fill_between(om_harm, 0, g_harm, alpha=0.35, color=C['sky'], label='Harmonic DOS')
    ax.plot(om_harm, g_harm, color=C['sky'], lw=1.0)

    for j, ((_, _, temp_label), vd) in enumerate(anh_entries):
        om = np.array(vd['omega_mev'])
        g  = np.array(vd['vdos'])
        col = [C['blue'], C['orange'], C['green']][j % 3]
        ax.plot(om, g, color=col, lw=1.2, label=f'Anharmonic VDOS ({temp_label})')

    ax.axvspan(0, cfg.VIB_ZERO_WINDOW_MEV, color=C['red'], alpha=0.15,
               label=f'Soft-mode window')
    ax.set_xlabel('ω (meV)')
    ax.set_ylabel('VDOS (normalised)')
    ax.set_title(f'{mat}/{sym} — Harmonic vs anharmonic VDOS')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False)

    plt.tight_layout()
    plt.savefig(f'vdos_comparison_{mat}_{sym}.pdf', dpi=150, bbox_inches='tight')
    plt.show()

## 6 — Gibbs free energy decomposition

In [ ]:
T_arr = np.array(HULL_TEMPERATURES, dtype=float)

for mat, sym in mat_sym_list:
    sym_dir = os.path.join(DATABASE_ROOT, mat, sym)

    # Check minimum data available
    if not os.path.exists(os.path.join(sym_dir, 'relaxation', 'energy.json')):
        print(f'[skip] {mat}/{sym}: no relaxation energy')
        continue

    # Build assembler
    asm = build_default_assembler(
        phonon_mode              = 'anharmonic',
        device                   = 'cpu',
        include_electronic       = True,
        include_magnetic         = True,
        include_configurational  = True,
    )
    result = asm.compute(sym_dir, T_arr)

    if result.get('E0_eV_per_atom') is None:
        print(f'[skip] {mat}/{sym}: GibbsAssembler returned no E0')
        continue

    E0      = result['E0_eV_per_atom']
    G       = np.array(result['G_eV_per_atom'])
    contribs = result.get('contributions', {})
    active   = result.get('available_contributions', [])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

    # (a) Stacked contributions
    contrib_colours = {
        'F_vib_anharmonic': C['blue'],
        'F_vib_harmonic'  : C['sky'],
        'F_el'            : C['orange'],
        'F_mag'           : C['purple'],
        'F_conf'          : C['green'],
        'PV'              : C['red'],
    }
    base = np.zeros_like(T_arr)
    for name, vals_list in contribs.items():
        vals = np.array(vals_list) * 1000  # eV → meV
        col  = contrib_colours.get(name, 'gray')
        ax1.fill_between(T_arr, base, base + vals,
                         alpha=0.7, color=col, label=f'$F_{{\\rm {name.replace("_", ",")}}}$')
        base = base + vals

    ax1.plot(T_arr, (G - E0) * 1000, color='black', lw=1.5, label='$G_{\\rm tot}$')
    ax1.axvline(1200, color='gray', lw=0.8, ls=':')
    ax1.set_xlabel('Temperature (K)')
    ax1.set_ylabel('$G - E_0$ (meV/atom)')
    ax1.set_title(f'(a) Free-energy contributions')
    ax1.legend(frameon=False, fontsize=6, ncol=2)
    ax1.set_xlim(0, max(HULL_TEMPERATURES))

    # (b) Thermodynamic integrals from the AIMD VDOS at each temperature
    anh_entries = [(k, v) for k, v in vdos_cache.items() if k[0] == mat and k[1] == sym]
    if anh_entries:
        for j, ((_, _, temp_label), vd) in enumerate(anh_entries):
            # Load thermo.json if available
            sim_dir    = os.path.join(sym_dir, 'aimd', temp_label)
            thermo_path = os.path.join(sim_dir, 'anharmonic_phonons', 'thermo.json')
            if os.path.exists(thermo_path):
                with open(thermo_path) as fh:
                    thermo = json.load(fh)
                T_md = thermo.get('temperature_K', 0)
                E    = thermo.get('E_vib_mev_atom', 0)
                F    = thermo.get('F_vib_mev_atom', 0)
                Cv   = thermo.get('Cv_mev_Katom', 0)
                S    = thermo.get('S_vib_mev_Katom', 0)
                col  = [C['blue'], C['orange'], C['green']][j % 3]
                ax2.scatter(T_md, E,  color=col, marker='o', s=60, label=f'{temp_label}: $E_{{vib}}$')
                ax2.scatter(T_md, F,  color=col, marker='s', s=60, label=f'{temp_label}: $F_{{vib}}$')

    ax2.set_xlabel('Temperature (K)')
    ax2.set_ylabel('Thermodynamic quantity (meV/atom)')
    ax2.set_title(f'(b) Vibrational thermo at MD temperatures')
    ax2.legend(frameon=False, fontsize=6)
    ax2.axhline(0, color='black', lw=0.5, ls='--', alpha=0.5)

    fig.suptitle(f'{mat}/{sym} — Gibbs free-energy analysis', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'gibbs_{mat}_{sym}.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: gibbs_{mat}_{sym}.pdf')
    print(f'  E0 = {E0:.4f} eV/atom')
    print(f'  G(1200 K) = {float(np.interp(1200, T_arr, G)):.4f} eV/atom')
    print(f'  Active contributions: {active}')

## 7 — Multi-temperature free-energy consistency check

If AIMD was run at multiple temperatures, this cell checks whether F(T) is
consistent across them. A large spread may indicate a phase transition.

In [ ]:
T_arr = np.array(HULL_TEMPERATURES, dtype=float)

for mat, sym in mat_sym_list:
    fe_agg = fe_cache.get((mat, sym))
    if fe_agg is None:
        continue

    F_std = np.array(fe_agg.get('F_std_eV_per_atom', np.zeros_like(T_arr))) * 1000  # meV
    F_mean = np.array(fe_agg.get('F_eV_per_atom', np.zeros_like(T_arr))) * 1000
    n_src  = fe_agg.get('n_sources', 1)

    if n_src < 2:
        print(f'{mat}/{sym}: only 1 AIMD temperature — no consistency check.')
        continue

    Ts = np.array(fe_agg.get('temperatures', HULL_TEMPERATURES), dtype=float)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

    # (a) F(T) with uncertainty band
    ax1.plot(Ts, F_mean, color=C['blue'], lw=1.5, label='$\\langle F(T) \\rangle$')
    ax1.fill_between(Ts, F_mean - F_std, F_mean + F_std,
                     alpha=0.35, color=C['blue'], label='±1σ across MD temps')
    ax1.set_xlabel('Temperature (K)')
    ax1.set_ylabel('F(T) (meV/atom)')
    ax1.set_title('(a) Averaged anharmonic F(T)')
    ax1.legend(frameon=False)

    # (b) Fractional spread
    safe_mean = np.where(np.abs(F_mean) > 1e-8, F_mean, 1.0)
    rel_spread = np.abs(F_std / safe_mean) * 100
    ax2.plot(Ts, rel_spread, color=C['red'], lw=1.2)
    ax2.axhline(cfg.THERMO_FE_CONSISTENCY_THRESHOLD * 100,
                color='orange', lw=1.0, ls='--',
                label=f'Warn threshold ({cfg.THERMO_FE_CONSISTENCY_THRESHOLD*100:.0f}%)')
    ax2.set_xlabel('Temperature (K)')
    ax2.set_ylabel('Fractional spread (%)')
    ax2.set_title('(b) F(T) consistency across MD temperatures')
    ax2.legend(frameon=False)

    if fe_agg.get('consistency_warning'):
        ax2.text(0.5, 0.95, '⚠ CONSISTENCY WARNING', transform=ax2.transAxes,
                 ha='center', va='top', color='red', fontsize=9, fontweight='bold')

    fig.suptitle(f'{mat}/{sym} — F(T) consistency  ({n_src} MD temperatures)',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'consistency_{mat}_{sym}.pdf', dpi=150, bbox_inches='tight')
    plt.show()

## 8 — Summary table of vibrational thermodynamics

In [ ]:
rows = []
for (mat, sym, temp_label), vd in vdos_cache.items():
    sim_dir    = os.path.join(DATABASE_ROOT, mat, sym, 'aimd', temp_label)
    thermo_path = os.path.join(sim_dir, 'anharmonic_phonons', 'thermo.json')
    if not os.path.exists(thermo_path):
        continue
    with open(thermo_path) as fh:
        t = json.load(fh)

    om   = np.array(vd['omega_mev'])
    g    = np.array(vd['vdos'])
    mask = om <= cfg.VIB_ZERO_WINDOW_MEV
    zeta = float(np.trapz(g[mask], om[mask])) if mask.any() else 0.0

    rows.append({
        'Material'       : mat,
        'Symmetry'       : sym,
        'T_MD (K)'       : int(t.get('temperature_K', 0)),
        'ν_mean (meV)'   : f"{t.get('nu_mean_mev', 0):.1f}",
        'E_vib (meV/at)' : f"{t.get('E_vib_mev_atom', 0):.1f}",
        'F_vib (meV/at)' : f"{t.get('F_vib_mev_atom', 0):.1f}",
        'Cv (meV/K/at)'  : f"{t.get('Cv_mev_Katom', 0):.3f}",
        'S_vib (meV/K/at)': f"{t.get('S_vib_mev_Katom', 0):.3f}",
        'ζ (%)'          : f"{zeta * 100:.2f}",
        'Vib status'     : 'FAIL' if zeta >= cfg.VIB_ZERO_FRAC_FAIL
                           else 'WARN' if zeta >= cfg.VIB_ZERO_FRAC_WARN
                           else 'OK',
    })

if rows:
    df = pd.DataFrame(rows)
    display(df.set_index(['Material', 'Symmetry', 'T_MD (K)']).style.set_caption(
        'Vibrational thermodynamics at MD simulation temperatures'
    ))
else:
    print('No thermo.json files found. Run notebook 1 first.')

## 9 — Export for paper figures

In [ ]:
# Regenerate all paper figures from real data (replaces synthetic versions)
from heatup.utils.plotting_utils import generate_all_figures

os.makedirs('paper_figures', exist_ok=True)
paths = generate_all_figures(output_dir='paper_figures')
print('\nPaper figures generated (synthetic data):')
for name, path in paths.items():
    print(f'  {name}: {path}')
print('\nTo use real data, replace the synthetic VDOS/hull inputs with vdos_cache and fe_cache above.')

---
**Done.** Results are available in:
- `vdos_<mat>_<sym>.pdf` — per-material VDOS panels
- `vdos_comparison_<mat>_<sym>.pdf` — harmonic vs anharmonic comparison
- `gibbs_<mat>_<sym>.pdf` — free-energy decomposition
- `consistency_<mat>_<sym>.pdf` — multi-temperature consistency
- `paper_figures/` — publication-ready figure set